# Example of initial setup of Weaviate

## Initial steps

In [ ]:
import os
import pandas as pd
from pathlib import Path

import weaviate
from weaviate.connect import ConnectionParams
from weaviate.classes.config import Configure, Property, DataType, Tokenization
from weaviate.classes.query import  MetadataQuery, Filter, Rerank, GroupBy, BM25Operator
from weaviate.classes.tenants import Tenant

from langchain.text_splitter import RecursiveCharacterTextSplitter

from functionality.agent import load_env

In [4]:
load_env()

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    print(client.is_ready())

## Loading data

Assume you have a CSV file named `data.csv` with the following columns:

- `title`
- `text`
- `category`

In [ ]:
df = pd.read_csv("path/to/data.csv", sep="$")
df.head()

In [ ]:
df.info()

## Determine the future chema

The code assumes that we will have 2 collection:

1. `Nodes` with 2 tenants: 
    - `kb_nodes` - to store already known full information
    - `graph_nodes` - to store information that we will discover during the nodes extrtaction
2. `Chunks` that will store chunks of text extracted from `Nodes` collection.

It's not optimal since `Chunks` is a duplication of `Nodes` but it will do for now.


## (Optional) Delete existing collections

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    client.collections.delete(
        "Nodes"
    )
    client.collections.delete(
        "Chunks"
    )

## Create collections

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    nodes_collection = client.collections.create(
        "Nodes",
        multi_tenancy_config=Configure.multi_tenancy(enabled=True),
        vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
            name="bge",
            vectorize_collection_name=False,
            source_properties=["title", "text", "category"]
        )],
        properties=[
            Property(
                name="title",
                data_type=DataType.TEXT,
                tokenization=Tokenization.FIELD
            ),
            Property(
                name="text",
                data_type=DataType.TEXT
            ),
            Property(
                name="category",
                data_type=DataType.TEXT,
                tokenization=Tokenization.FIELD
            )
        ])
    
    nodes_collection.tenants.create(
        tenants=[
            Tenant(name="kb_nodes"),
            Tenant(name="graph_nodes")
        ]
    )

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    chunks_collection = client.collections.create(
        "Chunks",
        vectorizer_config=[Configure.NamedVectors.text2vec_transformers(
            name="bge",
            vectorize_collection_name=False,
            source_properties=["title", "text", "category"]
        )],
        properties=[
            Property(
                name="title",
                data_type=DataType.TEXT,
                tokenization=Tokenization.FIELD
            ),
            Property(
                name="text",
                data_type=DataType.TEXT
            ),
            Property(
                name="chunk_number",
                data_type=DataType.INT,
                skip_vectorization=True
            ),
            Property(
                name="category",
                data_type=DataType.TEXT,
                tokenization=Tokenization.FIELD
            )
        ])

## Populate collections

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    nodes_collection = client.collections.get("Nodes")
    kb_nodes = nodes_collection.with_tenant("kb_nodes")
    chunk_collection = client.collections.get("Chunks")

    with chunk_collection.batch.dynamic() as batch:
        for i, row in df_exploded.iterrows():
            batch.add_object({
                    "title": row["title"],
                    "text": row["text_chunk"],
                    "category": row["category"],
                    "chunk_number":row["chunk_number"]
            }
        )
    with kb_nodes.batch.dynamic() as batch:
        for i, row in df.iterrows():
            batch.add_object({
                    "title": row["title"],
                    "text": row["text"],
                    "category": row["category"]
            }
        )
    
    graph_nodes = nodes_collection.with_tenant("graph_nodes")
    with graph_nodes.batch.dynamic() as batch:
        for i, row in df.iterrows():
            batch.add_object({
                    "title": row["title"],
                    "text": "",
                    "category": row["category"]
            }
        )

## Query collections

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    nodes_collection = client.collections.get("Chunks")
    response = nodes_collection.query.bm25(
        query="Key words to search",
        limit=5
    )
for o in response.objects:
    print(o.properties)

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    nodes_collection = client.collections.get("Chunks")
    response = nodes_collection.query.near_text(
        query="Some text to search",
        limit=5,
        rerank=Rerank(
            prop="text",
            query="important thing to emphasize",
        ),
    )
for o in response.objects:
    print(o.properties)

In [ ]:
with weaviate.connect_to_custom(
    http_host=os.getenv("WEAVIATE_HTTP_HOST", "localhost"),
    http_port=int(os.getenv("WEAVIATE_HTTP_PORT", "8080")),
    http_secure=os.getenv("WEAVIATE_HTTP_SECURE", "false").lower() == "true",
    grpc_host=os.getenv("WEAVIATE_GRPC_HOST", "localhost"),
    grpc_port=int(os.getenv("WEAVIATE_GRPC_PORT", "8081")),
    grpc_secure=os.getenv("WEAVIATE_GRPC_SECURE", "false").lower() == "true") as client:
    nodes_collection = client.collections.get("Chunks")
    response = nodes_collection.query.hybrid(
    query="Some text to search",
    alpha=0.5,
    limit=5
)
for o in response.objects:
    print(o.properties)